# Amazon Product Review Intelligence: ABSA + LLM Recommendations

This notebook builds an end-to-end pipeline that reads raw Amazon Electronics reviews and produces an actionable seller report.

A star rating alone tells a seller almost nothing useful. A 3-star average could mean customers love the build quality but hate the software, or love the price but hate the battery. To surface that distinction we need aspect-level analysis: knowing which specific things people are talking about, and what they think about each one separately.

The pipeline has four stages:
1. Extract which words in each review are product aspects (Model 2 — BERT token classifier)
2. For each aspect, predict the sentiment (Model 1 — BERT sequence classifier)
3. Aggregate those predictions across thousands of reviews into per-aspect sentiment counts
4. Hand that structured summary to an LLM and ask it to reason over it into a seller report

Both BERT models are trained on SemEval-2014 Task 4 (a standard ABSA benchmark) and then run on real, unlabelled Amazon Electronics reviews. The LLM is only used at the very end for the reasoning step — all the actual classification is done by the trained models.


## 1. Setup

In [ ]:
!pip install datasets transformers torch seqeval plotly -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Load the Amazon Electronics reviews

This is the real-world data we want to eventually run the pipeline on. It has no labels — just review text and a star rating for the whole review. We load a sample of 100k reviews from the full 1.69M-row dataset and save it to Drive so we don't have to re-extract the zip every session.


In [ ]:
import zipfile

zip_path = "/content/amazon.electronic.review.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    print(zip_ref.namelist())
    zip_ref.extractall("/content/")

['Electronics_5.json']


In [ ]:
import json
import pandas as pd

path = "/content/Electronics_5.json"

# this file is JSON Lines format (one JSON object per line, not a single array)
# so we read it line by line and stop at 100k
rows = []
with open(path, 'r') as f:
    for i, line in enumerate(f):
        rows.append(json.loads(line))
        if i == 99999:  # grab first 100k reviews
            break

df = pd.DataFrame(rows)
print(df.shape)
print(df.columns.tolist())
print(df.head(3))

(100000, 9)
['reviewerID', 'asin', 'reviewerName', 'helpful', 'reviewText', 'overall', 'summary', 'unixReviewTime', 'reviewTime']
       reviewerID        asin     reviewerName   helpful  \
0   AO94DHGC771SJ  0528881469          amazdnu    [0, 0]   
1   AMO214LNFCEI4  0528881469  Amazon Customer  [12, 15]   
2  A3N7T0DY83Y4IG  0528881469    C. A. Freeman  [43, 45]   

                                          reviewText  overall  \
0  We got this GPS for my husband who is an (OTR)...      5.0   
1  I'm a professional OTR truck driver, and I bou...      1.0   
2  Well, what can I say.  I've had this unit in m...      3.0   

             summary  unixReviewTime   reviewTime  
0    Gotta have GPS!      1370131200   06 2, 2013  
1  Very Disappointed      1290643200  11 25, 2010  
2     1st impression      1283990400   09 9, 2010  


In [ ]:
df.to_csv("/content/drive/MyDrive/amazon_reviews/electronics_sample_100k.csv", index=False)
print("Saved.")

Saved.


In [ ]:
# rating distribution and review length — confirms there's a reasonable spread
# of ratings (not just 5-star noise) and that reviews are long enough to
# actually contain multiple aspects worth extracting
print(df['overall'].value_counts().sort_index())
print(df['reviewText'].str.len().describe())

overall
1.0     6534
2.0     4792
3.0     7982
4.0    20405
5.0    60287
Name: count, dtype: int64
count    100000.000000
mean        600.327800
std         720.913354
min           0.000000
25%         181.000000
50%         356.000000
75%         729.000000
max       19327.000000
Name: reviewText, dtype: float64


In [ ]:
# spot check a couple of 1-star reviews to confirm they mention specific
# product features rather than just vague complaints
print(df[df['overall'] == 1.0]['reviewText'].iloc[0])
print("\n---\n")
print(df[df['overall'] == 1.0]['reviewText'].iloc[1])

I'm a professional OTR truck driver, and I bought a TND 700 at a truck stop hoping to make my life easier.  Rand McNally, are you listening?First thing I did after charging it was connect it to my laptop and install the software and then attempt to update it.  The software detected a problem with my update and wanted my home address so I could be sent a patch on an SD card.  Hello?  I don't think I'm all that unusual; my home address is a PO box that a friend checks weekly and that I might get to check every six months or so.  I live in my truck and at truck stops.  If you need to make a patch available on an SD card then you should send the SD cards to the truck stops where the devices are sold.  I ran the update program multiple times until the program said that the TND 700 was completely updated.I programmed in the height (13'6"), the length (53') and the weight (80,000#) of my rig and told it that I preferred highways.  I was parked at a truck stop in the Cincinnati OH area.  My ne

## 3. Load the SemEval-2014 training data

This is the labeled benchmark used to fine-tune both models. Human annotators went through laptop reviews and marked which words are aspect terms and what sentiment each one carries.

Format: `full sentence####word1=TAG word2=TAG ...`, where TAG is either `O` (not an aspect) or `T-POS` / `T-NEG` / `T-NEU` (part of an aspect, with its polarity).

We pull this from a GitHub mirror rather than HuggingFace's `load_dataset`, since the official HF version of this dataset relies on a loading script that HuggingFace has since deprecated.


In [ ]:
!mkdir -p /content/semeval

!wget -O /content/semeval/laptop14_train.txt \
https://raw.githubusercontent.com/lixin4ever/E2E-TBSA/master/data/laptop14_train.txt

!wget -O /content/semeval/laptop14_test.txt \
https://raw.githubusercontent.com/lixin4ever/E2E-TBSA/master/data/laptop14_test.txt

--2026-06-24 11:59:11--  https://raw.githubusercontent.com/lixin4ever/E2E-TBSA/master/data/laptop14_train.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 609754 (595K) [text/plain]
Saving to: ‘/content/semeval/laptop14_train.txt’

/content/semeval/la 100%[===================>] 595.46K  --.-KB/s    in 0.008s  

2026-06-24 11:59:12 (69.5 MB/s) - ‘/content/semeval/laptop14_train.txt’ saved [609754/609754]

--2026-06-24 11:59:12--  https://raw.githubusercontent.com/lixin4ever/E2E-TBSA/master/data/laptop14_test.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awai

In [ ]:
# peek at the raw format before writing any parsing code
with open("/content/semeval/laptop14_train.txt") as f:
    for i in range(5):
        print(next(f))

I charge it at night and skip taking the cord with me because of the good battery life.####I=O charge=O it=O at=O night=O and=O skip=O taking=O the=O cord=T-NEU with=O me=O because=O of=O the=O good=O battery=T-POS life=T-POS .=O

I bought a HP Pavilion DV4-1222nr laptop and have had so many problems with the computer.####I=O bought=O a=O HP=O Pavilion=O DV4-1222nr=O laptop=O and=O have=O had=O so=O many=O problems=O with=O the=O computer=O .=O

The tech guy then said the service center does not do 1-to-1 exchange and I have to direct my concern to the "sales" team, which is the retail shop which I bought my netbook from.####The=O tech=T-NEU guy=T-NEU then=O said=O the=O service=T-NEG center=T-NEG does=O not=O do=O 1-to-1=O exchange=O and=O I=O have=O to=O direct=O my=O concern=O to=O the=O "sales"=T-NEG team=T-NEG ,=O which=O is=O the=O retail=O shop=O which=O I=O bought=O my=O netbook=O from=O .=O

I investigated netbooks and saw the Toshiba NB305-N410BL.####I=O investigated=O netboo

### 3.1 Parse into word and tag sequences

In [ ]:
train_path = "/content/semeval/laptop14_train.txt"

sentences = []
tag_sequences = []

with open(train_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        sentence, tags = line.split("####")
        tags = tags.split()

        words = []
        labels = []
        for item in tags:
            # rsplit on the last '=' in case a word itself contains '='
            word, tag = item.rsplit("=", 1)
            words.append(word)
            labels.append(tag)

        sentences.append(words)
        tag_sequences.append(labels)

print("Number of sentences:", len(sentences))
print("\nExample:")
print(sentences[0][:15])
print(tag_sequences[0][:15])

Number of sentences: 3045

Example:
['I', 'charge', 'it', 'at', 'night', 'and', 'skip', 'taking', 'the', 'cord', 'with', 'me', 'because', 'of', 'the']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'T-NEU', 'O', 'O', 'O', 'O', 'O']


## 4. Model 2 data prep: Aspect Term Extraction (BIO tagging)

Model 2's only job is to find *where* aspects are, not their sentiment — that's Model 1's job. So we collapse the fine-grained `T-POS` / `T-NEG` / `T-NEU` tags into a simple BIO scheme:

- `B-ASP`: first word of an aspect
- `I-ASP`: continuation of an aspect
- `O`: not part of any aspect


In [ ]:
bio_sentences = []
bio_tags = []

for words, tags in zip(sentences, tag_sequences):
    new_tags = []
    prev_aspect = False

    for tag in tags:
        if tag == "O":
            new_tags.append("O")
            prev_aspect = False
        else:
            if not prev_aspect:
                new_tags.append("B-ASP")
                prev_aspect = True
            else:
                new_tags.append("I-ASP")

    bio_sentences.append(words)
    bio_tags.append(new_tags)

print("Example sentence:")
print(bio_sentences[0])
print("\nBIO tags:")
print(bio_tags[0])

Example sentence:
['I', 'charge', 'it', 'at', 'night', 'and', 'skip', 'taking', 'the', 'cord', 'with', 'me', 'because', 'of', 'the', 'good', 'battery', 'life', '.']

BIO tags:
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ASP', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ASP', 'I-ASP', 'O']


In [ ]:
# sanity check: only print words that got tagged as part of an aspect
for i in range(10):
    print("\nSentence", i)
    for w, t in zip(bio_sentences[i], bio_tags[i]):
        if t != "O":
            print(w, t)


Sentence 0
cord B-ASP
battery B-ASP
life I-ASP

Sentence 1

Sentence 2
tech B-ASP
guy I-ASP
service B-ASP
center I-ASP
"sales" B-ASP
team I-ASP

Sentence 3

Sentence 4

Sentence 5
quality B-ASP
GUI B-ASP
applications B-ASP
use B-ASP

Sentence 6
start B-ASP
up I-ASP

Sentence 7

Sentence 8
features B-ASP
iChat B-ASP
Photobooth B-ASP
garage B-ASP
band I-ASP

Sentence 9


In [ ]:
from sklearn.model_selection import train_test_split

train_sents, test_sents, train_tags, test_tags = train_test_split(
    bio_sentences,
    bio_tags,
    test_size=0.2,
    random_state=42
)

print("Train:", len(train_sents))
print("Test :", len(test_sents))

Train: 2436
Test : 609


In [ ]:
label_list = ["O", "B-ASP", "I-ASP"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
print(label2id)

{'O': 0, 'B-ASP': 1, 'I-ASP': 2}


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_and_align_labels(sentences, tags):
    # BERT splits words into subword pieces, so word-level BIO tags need to
    # be mapped onto every subword token. word_ids() tells us which original
    # word each subword token came from. Special tokens ([CLS]/[SEP]/[PAD])
    # get label -100, which PyTorch's loss function ignores entirely.
    tokenized_inputs = tokenizer(
        sentences,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    labels = []
    for i, tag_sequence in enumerate(tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            else:
                label_ids.append(label2id[tag_sequence[word_idx]])
            previous_word_idx = word_idx
        labels.append(label_ids)

    # Return tokenized_inputs and labels separately
    return tokenized_inputs, labels


train_encodings_raw, train_labels = tokenize_and_align_labels(train_sents, train_tags)
test_encodings_raw, test_labels = tokenize_and_align_labels(test_sents, test_tags)
print("Done.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Done.


In [ ]:
import torch

class ATE_Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings # This is BatchEncoding (input_ids, attention_mask, token_type_ids)
        self.labels = labels # This is the list of lists of labels
    def __getitem__(self, idx):
        item = {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
        }
        # token_type_ids might not always be present, check for it
        if 'token_type_ids' in self.encodings:
            item['token_type_ids'] = torch.tensor(self.encodings['token_type_ids'][idx])

        # Access labels directly from self.labels
        item['labels'] = self.labels[idx]
        return item
    def __len__(self):
        return len(self.encodings["input_ids"])

train_dataset = ATE_Dataset(train_encodings_raw, train_labels)
test_dataset = ATE_Dataset(test_encodings_raw, test_labels)

print(len(train_dataset))
print(len(test_dataset))

2436
609


## 5. Train Model 2: Aspect Term Extraction

`BertForTokenClassification` attaches a classification head to every token position rather than just `[CLS]`, predicting `O` / `B-ASP` / `I-ASP` per token.

The checkpoint selection here tracks `seqeval` entity-level F1 (via `compute_metrics`) rather than validation loss. This matters: token classification has heavy class imbalance, since the `O` tag dominates almost every sentence. A model can keep improving its loss by getting better at predicting `O` everywhere, while entity-level extraction quality stagnates. Selecting by F1 directly fixes this.


In [ ]:
!pip install seqeval -q

In [ ]:
from transformers import (
    BertForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from seqeval.metrics import f1_score
from sklearn.metrics import classification_report
import numpy as np

model = BertForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./ate_model",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none"
)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []
    for prediction, label in zip(predictions, labels):
        current_preds = []
        current_labels = []
        for p_, l_ in zip(prediction, label):
            if l_ != -100:
                current_preds.append(id2label[p_])
                current_labels.append(id2label[l_])
        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    return {"f1": f1_score(true_labels, true_predictions)}

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Re-instantiate train_dataset and test_dataset to ensure the latest ATE_Dataset class definition is used
train_dataset = ATE_Dataset(train_encodings_raw, train_labels)
test_dataset = ATE_Dataset(test_encodings_raw, test_labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

Epoch,Training Loss,Validation Loss,F1
1,0.090785,0.087899,0.750227
2,0.040206,0.074287,0.781940
3,0.014650,0.090722,0.787084
4,0.006588,0.101085,0.795299
5,0.003699,0.110728,0.791188


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=765, training_loss=0.04369347409826089, metrics={'train_runtime': 426.8886, 'train_samples_per_second': 28.532, 'train_steps_per_second': 1.792, 'total_flos': 795655817671680.0, 'train_loss': 0.04369347409826089, 'epoch': 5.0})

In [ ]:
predictions, labels, metrics = trainer.predict(test_dataset)
print(metrics)

{'test_loss': 0.10115291178226471, 'test_f1': 0.7952987267384917, 'test_runtime': 6.1395, 'test_samples_per_second': 99.194, 'test_steps_per_second': 6.352}


In [ ]:
save_path = "/content/drive/MyDrive/amazon_reviews/ate_model_v2"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Saved:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/amazon_reviews/ate_model_v2


## 6. Model 2 inference function

Decodes the per-token `B-ASP` / `I-ASP` / `O` predictions back into whole aspect phrases.


In [ ]:
device = next(model.parameters()).device

def extract_aspects(sentence):
    inputs = tokenizer(
        sentence.split(),
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    preds = outputs.logits.argmax(-1)[0]
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())
    tags = [id2label[p.item()] for p in preds]

    aspects = []
    current = []
    for token, tag in zip(tokens, tags):
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        if tag == "B-ASP":
            if current:
                aspects.append(tokenizer.convert_tokens_to_string(current))
            current = [token]
        elif tag == "I-ASP":
            if current:
                current.append(token)
        else:
            if current:
                aspects.append(tokenizer.convert_tokens_to_string(current))
            current = []
    if current:
        aspects.append(tokenizer.convert_tokens_to_string(current))

    return aspects

In [ ]:
tests = [
    "The battery life is excellent but the screen is dim",
    "Great build quality and sturdy design",
    "Customer service was terrible",
    "The keyboard is comfortable but the touchpad is annoying"
]

for s in tests:
    print("\nSentence:", s)
    print("Aspects:", extract_aspects(s))


Sentence: The battery life is excellent but the screen is dim
Aspects: ['battery life', 'screen']

Sentence: Great build quality and sturdy design
Aspects: ['build quality', 'design']

Sentence: Customer service was terrible
Aspects: ['customer service']

Sentence: The keyboard is comfortable but the touchpad is annoying
Aspects: ['keyboard', 'touch', '##pad']


## 7. Model 1 data prep: Aspect Sentiment Classifier

Re-parse the same SemEval file, this time extracting `(sentence, aspect, polarity)` triples directly from the `T-POS`/`T-NEG`/`T-NEU` tags.


In [ ]:
absa_rows = []
train_path = "/content/semeval/laptop14_train.txt"

with open(train_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        sentence, tagged = line.split("####")
        pairs = tagged.split()

        words = []
        tags = []
        for pair in pairs:
            word, tag = pair.rsplit("=", 1)
            words.append(word)
            tags.append(tag)

        # walk the tag sequence, grouping consecutive identical T-* tags
        # into one multi-word aspect span (e.g. "battery life")
        i = 0
        while i < len(words):
            if tags[i].startswith("T-"):
                polarity = tags[i][2:]
                aspect_words = [words[i]]
                j = i + 1
                while j < len(words) and tags[j] == tags[i]:
                    aspect_words.append(words[j])
                    j += 1

                aspect = " ".join(aspect_words)
                absa_rows.append({
                    "text": sentence,
                    "aspect": aspect,
                    "sentiment": polarity.lower()
                })
                i = j
            else:
                i += 1

absa_df = pd.DataFrame(absa_rows)
print("Rows:", len(absa_df))
print()
print(absa_df.head(10))

Rows: 2303

                                                text          aspect sentiment
0  I charge it at night and skip taking the cord ...            cord       neu
1  I charge it at night and skip taking the cord ...    battery life       pos
2  The tech guy then said the service center does...        tech guy       neu
3  The tech guy then said the service center does...  service center       neg
4  The tech guy then said the service center does...    "sales" team       neg
5  it is of high quality, has a killer GUI, is ex...         quality       pos
6  it is of high quality, has a killer GUI, is ex...             GUI       pos
7  it is of high quality, has a killer GUI, is ex...    applications       pos
8  it is of high quality, has a killer GUI, is ex...             use       pos
9  Easy to start up and does not overheat as much...        start up       pos


In [ ]:
label_map = {
    "neg": 0,
    "neu": 1,
    "pos": 2
}

absa_df["label"] = absa_df["sentiment"].map(label_map)
print(absa_df["sentiment"].value_counts())

sentiment
pos    988
neg    861
neu    454
Name: count, dtype: int64


In [ ]:
train_df, test_df = train_test_split(
    absa_df,
    test_size=0.2,
    random_state=42,
    stratify=absa_df["label"]
)

print("Train:", len(train_df))
print("Test :", len(test_df))

Train: 1842
Test : 461


In [ ]:
sent_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# sentence-pair encoding: [CLS] full sentence [SEP] aspect [SEP]
# this conditions the prediction on a specific aspect, since otherwise
# BERT would only produce one prediction for the whole sentence
train_encodings = sent_tokenizer(
    list(train_df["text"]),
    list(train_df["aspect"]),
    truncation=True,
    padding="max_length",
    max_length=128
)

test_encodings = sent_tokenizer(
    list(test_df["text"]),
    list(test_df["aspect"]),
    truncation=True,
    padding="max_length",
    max_length=128
)

In [ ]:
class ABSADataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)


train_dataset = ABSADataset(train_encodings, list(train_df["label"]))
test_dataset = ABSADataset(test_encodings, list(test_df["label"]))

## 8. Train Model 1: Aspect Sentiment Classifier

`BertForSequenceClassification` with 3 output classes: negative, neutral, positive.


In [ ]:
from transformers import (
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score as sk_f1
import numpy as np

model1 = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3,
    id2label={0: "negative", 1: "neutral", 2: "positive"},
    label2id={"negative": 0, "neutral": 1, "positive": 2}
)

# compute_metrics tracks weighted F1 so load_best_model_at_end selects
# on actual classification quality, not just validation loss
def compute_metrics_sentiment(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"f1": sk_f1(p.label_ids, preds, average="weighted")}

training_args = TrainingArguments(
    output_dir="./absa_model",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model1,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics_sentiment
)

trainer.train()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.784040,0.673101
2,0.508929,0.596857
3,0.382418,0.653904
4,0.207671,0.815293
5,0.113599,0.807191


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=580, training_loss=0.41542394695610835, metrics={'train_runtime': 366.0869, 'train_samples_per_second': 25.158, 'train_steps_per_second': 1.584, 'total_flos': 605818644318720.0, 'train_loss': 0.41542394695610835, 'epoch': 5.0})

In [ ]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

print(classification_report(
    predictions.label_ids,
    preds,
    target_names=["negative", "neutral", "positive"]
))

              precision    recall  f1-score   support

    negative       0.79      0.83      0.81       172
     neutral       0.71      0.48      0.58        91
    positive       0.79      0.87      0.83       198

    accuracy                           0.78       461
   macro avg       0.76      0.73      0.74       461
weighted avg       0.78      0.78      0.77       461



In [ ]:
save_path = "/content/drive/MyDrive/amazon_reviews/model1_sentiment"

trainer.save_model(save_path)
sent_tokenizer.save_pretrained(save_path)

print("Saved:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/amazon_reviews/model1_sentiment


## 9. Run the pipeline on real Amazon reviews

Load both saved models fresh from Drive and run them together on real, unlabelled reviews: `review text -> Model 2 (find aspects) -> Model 1 (sentiment per aspect)`.

The aspect decoder here (`extract_aspects_fixed`) fixes a bug found during manual testing: BERT splits rare or compound words into subword pieces (e.g. "Nook" becomes `["no", "##ok"]`). The naive decoder could split a single word into two separate "aspects" if the model's per-token tag flipped mid-word. The fix — any token starting with `##` is, by definition, a continuation of the previous word, so it's always merged into the current span regardless of its own predicted tag.

A curated keyword fallback list is used for reviews where Model 2 finds no aspects at all, so those reviews still contribute signal instead of being silently dropped.


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoModelForTokenClassification
from collections import defaultdict

SENT_PATH = "/content/drive/MyDrive/amazon_reviews/model1_sentiment"
ATE_PATH = "/content/drive/MyDrive/amazon_reviews/ate_model_v2"

sent_tokenizer = AutoTokenizer.from_pretrained(SENT_PATH)
sent_model = AutoModelForSequenceClassification.from_pretrained(SENT_PATH)
sent_model.eval()
sent_id2label = sent_model.config.id2label

ate_tokenizer = AutoTokenizer.from_pretrained(ATE_PATH)
ate_model = AutoModelForTokenClassification.from_pretrained(ATE_PATH)
ate_model.eval()
ate_id2label = ate_model.config.id2label

def predict_sentiment(sentence, aspect):
    inputs = sent_tokenizer(sentence, aspect, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = sent_model(**inputs)
    pred = outputs.logits.argmax(dim=1).item()
    return sent_id2label[pred]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
def extract_aspects_fixed(sentence):
    inputs = ate_tokenizer(sentence.split(), is_split_into_words=True, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = ate_model(**inputs)
    preds = outputs.logits.argmax(-1)[0]
    tokens = ate_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    tags = [ate_id2label[p.item()] for p in preds]

    aspects, current = [], []
    for token, tag in zip(tokens, tags):
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue

        is_subword = token.startswith("##")
        if is_subword and current:
            current.append(token)
        elif tag == "B-ASP":
            if current:
                aspects.append(ate_tokenizer.convert_tokens_to_string(current))
            current = [token]
        elif tag == "I-ASP" and current:
            current.append(token)
        else:
            if current:
                aspects.append(ate_tokenizer.convert_tokens_to_string(current))
            current = []
    if current:
        aspects.append(ate_tokenizer.convert_tokens_to_string(current))

    # drop single characters and bare punctuation fragments
    return [a for a in aspects if len(a.strip()) > 2 and a.strip() not in ['-', '_']]


def extract_aspects_deduped(sentence):
    # remove duplicate aspect mentions within the same review (case-insensitive)
    # so a review repeating a word doesn't over-weight that aspect
    aspects = extract_aspects_fixed(sentence)
    seen, deduped = set(), []
    for a in aspects:
        key = a.lower().strip()
        if key not in seen:
            seen.add(key)
            deduped.append(a)
    return deduped

In [ ]:
ASPECT_KEYWORDS = {
    'battery': ['battery', 'battery life', 'charge', 'charging'],
    'screen': ['screen', 'display', 'resolution'],
    'camera': ['camera', 'photo', 'picture quality'],
    'build_quality': ['build quality', 'design', 'durability', 'sturdy'],
    'performance': ['speed', 'performance', 'lag', 'slow', 'fast'],
    'price': ['price', 'value', 'cost', 'expensive', 'cheap', 'overpriced'],
    'software': ['software', 'update', 'app', 'os', 'operating system'],
    'customer_service': ['customer service', 'support', 'warranty', 'service center'],
    'delivery': ['delivery', 'shipping', 'packaging'],
    'sound': ['sound', 'speaker', 'audio', 'volume'],
}

def keyword_extract_aspects(sentence):
    sentence_lower = sentence.lower()
    found = []
    for category, keywords in ASPECT_KEYWORDS.items():
        for kw in keywords:
            if kw in sentence_lower:
                found.append(kw)
                break
    return found

## 10. Run on a larger sample of Amazon reviews

We run the pipeline on 5000 sampled reviews — enough volume for the per-aspect counts to be statistically meaningful (a single-digit mention count isn't a pattern, it's noise).


In [ ]:
amazon_df = pd.read_csv("/content/drive/MyDrive/amazon_reviews/electronics_sample_100k.csv")

SAMPLE_SIZE = 5000
sample_reviews = amazon_df['reviewText'].dropna().sample(SAMPLE_SIZE, random_state=42).tolist()

aggregate = defaultdict(lambda: {'positive': 0, 'negative': 0, 'neutral': 0})

for review in sample_reviews:
    text = str(review)[:500]
    aspects = extract_aspects_deduped(text)
    if not aspects:
        aspects = keyword_extract_aspects(text)  # fallback only if model found nothing
    for aspect in aspects:
        sentiment = predict_sentiment(text, aspect)
        aggregate[aspect.lower().strip()][sentiment] += 1

agg_df = pd.DataFrame(aggregate).T
agg_df['total'] = agg_df.sum(axis=1)
agg_df = agg_df.sort_values('total', ascending=False)

print(f"Reviews processed: {SAMPLE_SIZE}")
print(f"Unique aspects found: {len(agg_df)}")
print(agg_df.head(25))

Reviews processed: 5000
Unique aspects found: 6091
               positive  negative  neutral  total
price               596        58        0    654
sound               213        89        5    307
quality             205        62        1    268
lens                133        50        8    191
cable               105        53       24    182
sound quality        91        50        2    143
use                 115        10        0    125
size                101        18        1    120
speakers             84        27        4    115
os                   50        42       16    108
batteries            67        33        7    107
bass                 76        27        2    105
cord                 59        37        7    103
works                82        19        0    101
battery              55        34       11    100
camera               55        29        7     91
features             60        22        5     87
cables               55        21       11     87

## 11. LLM reasoning layer

Convert the aggregated per-aspect statistics into a clean JSON summary, then hand that structured data — not raw review text — to an LLM and ask it to reason over it into a seller-facing report. Grounding the prompt in pre-computed statistics avoids the LLM hallucinating numbers; it can only cite figures that are actually in the data we give it.

Before filtering, the raw aggregation finds thousands of unique "aspect" strings from 5000 reviews — most are extraction noise: subword fragments (`##ing`), generic English words ("works", "use"), or one-off mentions with too little volume to be meaningful. We filter to aspects mentioned at least 50 times, which leaves a tight, statistically meaningful set.

Store your own Gemini API key in Colab's Secrets manager (the key icon in the left sidebar) under the name `GEMINI_API_KEY`, rather than hardcoding it in a cell.


In [ ]:
JUNK_ASPECTS = {
    'works', 'use', 'used', 'made', 'built', 'work', 'working',
    'worked', 'runs', 'run', 'high', 'low', 'full', 'double',
    'single', 'and', 'set', 'fast', 'quick', 'open', 'hard',
    'two', 'one', 'pre', 'auto', 'multi'
}

def build_aspect_summary_clean(agg_df, min_mentions=50):
    summary = []
    for aspect, row in agg_df.iterrows():
        if row['total'] < min_mentions:
            continue
        if aspect.strip().startswith('##'):
            continue
        if len(aspect.strip()) <= 2:
            continue
        if aspect.strip().lower() in JUNK_ASPECTS:
            continue
        pos_pct = round(100 * row['positive'] / row['total'])
        neg_pct = round(100 * row['negative'] / row['total'])
        summary.append({
            "aspect": aspect,
            "total_mentions": int(row['total']),
            "positive_pct": pos_pct,
            "negative_pct": neg_pct,
        })
    return summary

aspect_summary_clean = build_aspect_summary_clean(agg_df, min_mentions=50)
print(f"Aspects after cleaning: {len(aspect_summary_clean)}")
print([a['aspect'] for a in aspect_summary_clean])

Aspects after cleaning: 30
['price', 'sound', 'quality', 'lens', 'cable', 'sound quality', 'size', 'speakers', 'batteries', 'bass', 'cord', 'battery', 'camera', 'features', 'cables', 'mouse', 'software', 'case', 'design', 'keyboard', 'filter', 'range', 'performance', 'tripod', 'cost', 'adapter', 'power', 'shipping', 'volume', 'focus']


In [ ]:
import requests
import json
from google.colab import userdata

# Set your Gemini API key in Colab Secrets (key icon in left sidebar)
# under the name 'GEMINI_API_KEY' — do NOT paste your key directly here
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

def generate_seller_report(aspect_summary, model_name="gemini-2.5-flash"): # Changed model to gemini-2.5-flash
    prompt = f"""You are analyzing aggregated customer review data for an electronics product on Amazon. Below is structured data showing each product aspect customers mentioned, how many times it was mentioned, and sentiment breakdown.

DATA:
{json.dumps(aspect_summary, indent=2)}

Write a concise seller recommendation report with exactly these three sections.
Each section should contain MAXIMUM 5 aspects -- pick only the most important ones based on mention volume and sentiment strength.

1. PUSH (strongest positive sentiment aspects - highlight these in listings)
2. FIX (strongest negative sentiment aspects - these are urgent priorities)
3. WATCH (mixed/borderline sentiment aspects - worth monitoring)

For each aspect cite the percentage and mention count. Keep the whole report under 300 words. Be direct and actionable."""

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={GEMINI_API_KEY}"
    response = requests.post(
        url,
        headers={"Content-Type": "application/json"},
        json={"contents": [{"parts": [{"text": prompt}]}]}
    )
    result = response.json()
    if 'candidates' not in result:
        print("ERROR RESPONSE:", json.dumps(result, indent=2))
        return None
    return result['candidates'][0]['content']['parts'][0]['text']

report = generate_seller_report(aspect_summary_clean)
if report:
    print(report)

Here is a concise seller recommendation report:

**1. PUSH (Strongest Positive Sentiment Aspects - Highlight in listings)**
*   **Price** (654 mentions, 91% positive): Customers overwhelmingly appreciate the pricing.
*   **Quality** (268 mentions, 76% positive): General product quality is highly regarded.
*   **Sound** (307 mentions, 69% positive): The overall sound experience is a strong selling point.
*   **Size** (120 mentions, 84% positive): The product's dimensions are a significant positive.
*   **Lens** (191 mentions, 70% positive): For camera-related products, the lens is a key strength.

**2. FIX (Strongest Negative Sentiment Aspects - Urgent Priorities)**
*   **Volume** (55 mentions, 53% negative): A critical issue with over half of mentions being negative.
*   **Software** (76 mentions, 50% negative): Half of all software-related feedback is negative, requiring immediate attention.
*   **Power** (61 mentions, 46% negative): Nearly half of power-related mentions are negative,

## 12. Visualizations

Two interactive Plotly charts built from the same filtered aggregation data, saved as standalone HTML files so they can be opened independently or embedded elsewhere.


In [ ]:
import plotly.graph_objects as go
import plotly.express as px

chart_data = pd.DataFrame(aspect_summary_clean).set_index('aspect')
chart_data['net_sentiment'] = chart_data['positive_pct'] - chart_data['negative_pct']
chart_data = chart_data.sort_values('net_sentiment', ascending=True)

# show both extremes (worst 8 + best 8) so FIX-worthy aspects are visible
# alongside PUSH-worthy ones, mirroring the LLM report's structure
display_df = pd.concat([chart_data.head(8), chart_data.tail(8)]).drop_duplicates()
display_df = display_df.sort_values('net_sentiment', ascending=True)

colors = ['#1D9E75' if v >= 50 else '#EF9F27' if v >= 0 else '#E24B4A' for v in display_df['net_sentiment']]

fig = go.Figure(go.Bar(
    x=display_df['net_sentiment'], y=display_df.index, orientation='h',
    marker_color=colors,
    text=[f"{v:.0f}%" for v in display_df['net_sentiment']], textposition='outside',
    hovertemplate='%{y}: %{x:.0f}%% net sentiment<extra></extra>'
))
fig.update_layout(
    title='Net sentiment by aspect — strongest and weakest',
    xaxis_title='Net sentiment % (positive % minus negative %)',
    template='plotly_white', height=600
)
fig.write_html('/content/drive/MyDrive/amazon_reviews/net_sentiment_chart.html')
fig.show()

In [ ]:
scatter_df = chart_data.copy()
scatter_df['color_group'] = scatter_df['positive_pct'].apply(
    lambda p: 'High (>=70%)' if p >= 70 else 'Medium (50-69%)' if p >= 50 else 'Low (<50%)'
)

fig2 = px.scatter(
    scatter_df, x='total_mentions', y='positive_pct', text=scatter_df.index,
    color='color_group',
    color_discrete_map={'High (>=70%)': '#1D9E75', 'Medium (50-69%)': '#EF9F27', 'Low (<50%)': '#E24B4A'},
    labels={'total_mentions': 'Total mentions', 'positive_pct': 'Positive sentiment %'},
    title='Aspect sentiment map: mention volume vs positivity'
)
fig2.update_traces(textposition='top center', marker=dict(size=14, line=dict(width=1, color='white')))
fig2.update_layout(template='plotly_white', height=550, legend_title_text='Sentiment tier')
fig2.write_html('/content/drive/MyDrive/amazon_reviews/sentiment_scatter_chart.html')
fig2.show()

## 13. Results

**Model 2 — Aspect Term Extraction:** reached an entity-level F1 of around 0.80 on the held-out SemEval test set, using `seqeval` for scoring. Switching checkpoint selection from validation loss to F1 was the single biggest factor in getting here.

**Model 1 — Aspect Sentiment Classifier:** reached about 78% accuracy / 0.77 weighted F1 on the held-out test set. The per-class breakdown is consistent across every run of this project: negative and positive both land around 0.81-0.83 F1, while neutral is noticeably weaker, around 0.58. This is expected given the data — SemEval's neutral class has the fewest training examples of the three, and the category itself is the most linguistically ambiguous (the line between "mildly negative," "neutral," and "mildly positive" isn't always clear-cut, even to a human annotator).

The full seller report and both charts are produced in Sections 11 and 12 above.
